# Machine Learning (part 2) - Model Optimization

### Advanced ML (optional for your project!)  

# Review & Context

## Last lecture (ML 1)

- ✓ What is Machine Learning?
- ✓ Supervised Learning (Classification vs. Regression)
- ✓ Train/Test Split
- ✓ Trained first models (NCC, KNN, Decision Tree, etc.)
- ✓ Accuracy as first metric

## in this lecture

**Overall Goal: Evaluate, compare, and optimize models**

# Problem: Accuracy Alone Is Not Enough!

## Example: Medical Diagnosis

**Scenario:** Detect disease (1% of patients are sick)

```python
# "Dumb" model: Always predict "healthy"
def dummy_model(patient):
    return 0  # always healthy

# Accuracy = 99%! (because 99% are actually healthy)
```

**But:** The model doesn't detect a single sick person! 🚨

---

> **Accuracy can be misleading with imbalanced datasets!**

# Understanding the Confusion Matrix

## The 4 Cases of Binary Classification

|  | **Predicted: Negative** | **Predicted: Positive** |
|---|---|---|
| **Actual: Negative** | True Negative (TN) ✅ | False Positive (FP) ❌ |
| **Actual: Positive** | False Negative (FN) ❌ | True Positive (TP) ✅ |

## Medical Example

- **TP:** Sick recognized as sick ✅
- **TN:** Healthy recognized as healthy ✅
- **FP:** Healthy recognized as sick (false alarm) ❌
- **FN:** Sick recognized as healthy (dangerous!) 🚨

# Confusion Matrix Visualized

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Make predictions
y_pred = model.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                               display_labels=['Healthy', 'Sick'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()
```

**Interpretation:** Diagonal = Correct predictions!

# Precision vs. Recall

## Two Important Metrics

### **Precision**
$$\text{Precision} = \frac{TP}{TP + FP}$$

*"Of all cases predicted as positive, how many are actually positive?"*

### **Recall (Sensitivity, True Positive Rate)**
$$\text{Recall} = \frac{TP}{TP + FN}$$

*"Of all actually positive cases, how many did we find?"*

# Precision vs. Recall: Trade-off

## Spam Filter Example

| Filter Type | Precision | Recall |
|----------|-----------|--------|
| **Strict Filter** | High ✅ | Low ❌ |
| (obvious spam only) | Few false positives | Much spam let through |
| **Loose Filter** | Low ❌ | High ✅ |
| (aggressive filtering) | Many false alarms | Almost no spam let through |

---

> **Trade-off:** Precision ↑ = Recall ↓ and vice versa

# ROC Curve & AUC

## Receiver Operating Characteristic

**ROC Curve:** Visualizes the trade-off between True Positive Rate and False Positive Rate

**AUC (Area Under Curve):** Area under the ROC curve
- AUC = 1.0: Perfect model
- AUC = 0.5: Random guessing
- AUC < 0.5: Worse than random

```python
from sklearn.metrics import roc_auc_score, RocCurveDisplay

# Calculate ROC-AUC
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

# Plot ROC curve
RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title(f'ROC Curve (AUC = {auc:.3f})')
```

# F1-Score: The Best of Both Worlds

## Harmonic Mean of Precision and Recall

$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

## Properties

- Combines Precision and Recall
- Well-suited for imbalanced datasets
- Range: 0 to 1 (higher = better)
- Penalizes extreme imbalances between Precision and Recall

```python
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))
```

# Metrics Overview

## When to Use Which Metric?

| Metric | When to use? | Example |
|--------|----------------|----------|
| **Accuracy** | Balanced classes | Predicting a coin flip |
| **Precision** | False positives are costly | Spam filter (don't lose important emails) |
| **Recall** | False negatives are costly | Cancer diagnosis (don't miss sick patients) |
| **F1-Score** | Imbalanced data | Fraud detection |
| **ROC-AUC** | Threshold-independent | Model comparison |

# Problem with Train/Test Split

## What if We're Unlucky?

```python
# Random split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42  # What if random_state were different?
)
```

**Problem:**
- Result depends on chance
- Difficult samples might all end up in the test set
- Small dataset: evaluation depends heavily on which samples end up in the test set

---

> **Solution:** Cross-Validation!

# Cross-Validation: Robust Evaluation

## K-Fold Cross-Validation

**Idea:** Split data into K parts (folds), train K times

```
Fold 1: [Test] [Train] [Train] [Train] [Train]
Fold 2: [Train] [Test] [Train] [Train] [Train]
Fold 3: [Train] [Train] [Test] [Train] [Train]
Fold 4: [Train] [Train] [Train] [Test] [Train]
Fold 5: [Train] [Train] [Train] [Train] [Test]
```

**Advantage:**
- Each sample is used once for testing
- More robust estimate of model performance
- Less dependent on chance

# Cross-Validation in sklearn

```python
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Define model
model = DecisionTreeClassifier(max_depth=5, random_state=42)

# 5-Fold Cross-Validation
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"Scores: {scores}")
print(f"Mean Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")
```

**Typical K values:** 5 or 10

**Alternative metrics:**
```python
scoring='f1'       # F1-Score
scoring='roc_auc'  # ROC-AUC
scoring='precision' # Precision
```

# Hyperparameter Tuning

## What Are Hyperparameters?

**Parameters:** Learned by the model (e.g. weights)

**Hyperparameters:** Set BEFORE training (e.g. max_depth)

## Examples

| Algorithm | Important Hyperparameters |
|-------------|------------------------|
| Decision Tree | `max_depth`, `min_samples_split` |
| Random Forest | `n_estimators`, `max_depth` |
| KNN | `n_neighbors` |
| SVM | `C`, `kernel`, `gamma` |

# Grid Search: Systematic Tuning

## Try All Combinations

```python
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Define parameter grid
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Grid Search with Cross-Validation
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1  # Parallel computation
)

# Training
grid_search.fit(X_train, y_train)

# Best parameters
print(f"Best params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.3f}")
```

# Grid Search: Analyzing Results

```python
# All results as DataFrame
results = pd.DataFrame(grid_search.cv_results_)

# Top 5 combinations
results[['params', 'mean_test_score', 'std_test_score']] \
    .sort_values('mean_test_score', ascending=False) \
    .head()

# Use best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Final evaluation
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))
```

# Feature Engineering

## Better Features = Better Models

### 1. Feature Creation (Create new features)
```python
# Example: Interactions
df['age_times_bmi'] = df['age'] * df['bmi']

# Polynomial features
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
```

### 2. Feature Transformation
```python
# Logarithmic transformation
df['log_price'] = np.log1p(df['price'])

# Binning
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 100], 
                          labels=['young', 'middle', 'old'])
```

# Feature Scaling

## Why Important?

Some algorithms (KNN, SVM, Logistic Regression) are sensitive to feature scales

### Standardization (StandardScaler)
```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Only transform, never fit!
```

**Result:** Mean = 0, Std = 1

### Normalization (MinMaxScaler)
```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
```

**Result:** Values between 0 and 1

# Feature Selection Revisited

## Three Approaches

### 1. Correlation-based
```python
# Features with high correlation to target
correlation = df.corr()['target'].abs().sort_values(ascending=False)
top_features = correlation.head(10).index.tolist()
```

### 2. Model-based (Feature Importance)
```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
```

# Feature Selection: SelectKBest

## Automatic Selection of Best Features

```python
from sklearn.feature_selection import SelectKBest, f_classif, f_classif

# Select top 10 features
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Which features were selected?
selected_features = X_train.columns[selector.get_support()]
print(f"Selected features: {selected_features.tolist()}")

# Train model with selected features
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_selected, y_train)
```

# Pipeline: Everything Together

## Elegant Workflow with sklearn Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=10)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Training
pipeline.fit(X_train, y_train)

# Prediction
y_pred = pipeline.predict(X_test)

# Everything in one step!
```

# Pipeline with GridSearchCV

## Hyperparameter Tuning for the Entire Pipeline

```python
param_grid = {
    'selector__k': [5, 10, 15],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.3f}")
```

# The Complete ML Workflow

## 7 Steps to an Optimal Prediction

1. **Load & explore data** (EDA)
2. **Clean data** (missing values, outliers)
3. **Feature Engineering** (new features, transformations)
4. **Train/Test Split** (or Cross-Validation)
5. **Train & compare models** (multiple algorithms)
6. **Hyperparameter Tuning** (GridSearchCV)
7. **Final Evaluation** (Confusion Matrix, Precision, Recall, F1, ROC-AUC)

---

> **Iteration is key:** Go back to step 3 if needed!

# Summary

## What We Learned Today

✅ **Evaluation Metrics:** Precision, Recall, F1-Score, ROC-AUC  
✅ **Confusion Matrix:** Understanding True/False Positives/Negatives  
✅ **Cross-Validation:** Robust model evaluation  
✅ **Hyperparameter Tuning:** GridSearchCV for optimal parameters  
✅ **Feature Engineering:** Better features = better models  
✅ **Pipeline:** Elegant end-to-end workflow  

# optional tasks for your project

1. **Evaluation:** Create Confusion Matrix + Classification Report
2. **Cross-Validation:** Evaluate your model with 5-Fold CV
3. **Hyperparameter Tuning:** Use GridSearchCV
4. **Feature Engineering:** Create at least 2 new features (optional)
5. **Comparison:** Document which model performs best and why

---

**Next Week:** Streamlit – Interactive Dashboards!

---


# Resources

## 📚 Documentation
- [sklearn Model Selection](https://scikit-learn.org/stable/model_selection.html)
- [sklearn Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)

## 🎓 Tutorials
- [Kaggle: Model Validation](https://www.kaggle.com/learn/intermediate-machine-learning)
- [scikit-learn Cross-validation Guide](https://scikit-learn.org/stable/modules/cross_validation.html)

## 📖 Books
- "Hands-On Machine Learning" - Chapters 2 & 3 (Géron)
- "Introduction to Statistical Learning" - Chapter 5 (James et al.)